In [84]:
import copy
import os
import argparse
from erasure.utils.logger import GLogger
import torch
import numpy as np
import random
from erasure.utils.config.local_ctx import Local
from erasure.utils.config.global_ctx import Global, bcolors 
from erasure.core.factory_base import ConfigurableFactory
from erasure.data.datasets.DatasetManager import DatasetManager

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [85]:
config_file = os.path.join("configs","LinkAttack", "Citeseer_edge.jsonc")

In [86]:
global_ctx = Global(config_file)
global_ctx.factory = ConfigurableFactory(global_ctx)

#Create Dataset
data_manager = global_ctx.factory.get_object( Local( global_ctx.config.data ))
global_ctx.dataset = data_manager

@,-748268051 | INFO | 3517115 - Creating Global Context for: configs/LinkAttack/Citeseer_edge.jsonc
�<9�/,-748268048 | INFO | 3517115 - Setting seeds to: 0
!,-748268038 | INFO | 3517115 - REMOVAL TYPE SET AS edge
,-748268037 | INFO | 3517115 - Caching System: False.
����0,-748268037 | INFO | 3517115 - Instantiating: torch_geometric.datasets.Planetoid


`�sO/,-748267887 | INFO | 3517115 - Created Configurable: erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource
`�z8/,-748267886 | INFO | 3517115 - {'class': 'erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource', 'parameters': {'datasource': {'class': 'torch_geometric.datasets.Planetoid', 'parameters': {'root': 'resources/data', 'name': 'Citeseer'}, 'preprocess': []}}}
����0,-748267850 | INFO | 3517115 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
pp��0,-748267799 | INFO | 3517115 - ['all', 'all_shuffled', '-']
����0,-748267798 | INFO | 3517115 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
��t8/,-748267789 | INFO | 3517115 - ['all', 'all_shuffled', '-', 'train_0', 'test']
����0,-748267788 | INFO | 3517115 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
��t8/,-748267788 | INFO | 3517115 - ['all', 'all_shuffled', '-', 'train_0', 'test',

In [87]:
#Create Predictor
current = Local(global_ctx.config.predictor)
current.dataset = data_manager
predictor = global_ctx.factory.get_object(current)
global_ctx.predictor = predictor
global_ctx.logger.info('Global Predictor: ' + str(predictor))

����0,-748267699 | INFO | 3517115 - Instantiating: erasure.model.graphs.GCN.GCN


����0,-748267691 | INFO | 3517115 - Instantiating: torch.optim.Adam
����0,-748267690 | INFO | 3517115 - Instantiating: torch.nn.CrossEntropyLoss
graph has edges  torch.Size([2, 9104])
��t,-748267608 | INFO | 3517115 - epoch = 0 ---> loss = 1.7978	 accuracy = 0.1720
graph has edges  torch.Size([2, 9104])
��t,-748267557 | INFO | 3517115 - epoch = 1 ---> loss = 1.7470	 accuracy = 0.3599
graph has edges  torch.Size([2, 9104])
��t,-748267505 | INFO | 3517115 - epoch = 2 ---> loss = 1.6973	 accuracy = 0.5436
graph has edges  torch.Size([2, 9104])
��t,-748267455 | INFO | 3517115 - epoch = 3 ---> loss = 1.6504	 accuracy = 0.6200
graph has edges  torch.Size([2, 9104])
��t,-748267395 | INFO | 3517115 - epoch = 4 ---> loss = 1.5992	 accuracy = 0.6785
graph has edges  torch.Size([2, 9104])
��t,-748267325 | INFO | 3517115 - epoch = 5 ---> loss = 1.5530	 accuracy = 0.7027
graph has edges  torch.Size([2, 9104])
��t,-748267262 | INFO | 3517115 - epoch = 6 ---> loss = 1.5004	 accuracy = 0.7219
graph 

In [88]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [89]:
data_manager.partitions['all'][0][1]

tensor([3, 1, 5,  ..., 3, 1, 5])

In [90]:
import torch
print(torch.version.cuda)

12.6


In [91]:
print(predictor.optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.0005000000000000002
    maximize: False
    weight_decay: 0
)


In [92]:
import torch
print(torch.__version__)
print(torch.version.cuda)


2.7.0+cu126
12.6


In [93]:
#Create unlearners 
unlearners = []
unlearners_cfg = global_ctx.config.unlearners
for un in unlearners_cfg:
    current = Local(un)
    current.dataset = data_manager
    current.predictor = copy.deepcopy(predictor)
    unlearners.append( global_ctx.factory.get_object(current) )

`�sO/,-748264394 | INFO | 3517115 - Created Configurable: erasure.unlearners.certified_graph_unlearners.IDEA.IDEA


In [94]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [95]:
data_manager.partitions['all'][0][0].edge_index

tensor([[   0,    1,    1,  ..., 3324, 3325, 3326],
        [ 628,  158,  486,  ..., 2820, 1643,   33]])

In [96]:
data_manager.partitions['all'].revise_graph_edges([(0,1),(0,2)])[0][0]

Data(x=[3327, 3703], edge_index=[2, 0])

In [97]:
print(len( data_manager.partitions['forget']) )

1820


In [98]:
#Evaluator
current = Local(global_ctx.config.evaluator)
current.unlearners = unlearners
evaluator = global_ctx.factory.get_object(current)

# Evaluations
for unlearner in unlearners:
    global_ctx.logger.info(f'''{bcolors.OKGREEN}####\t\t Evaluating Unlearner {unlearner.__class__.__name__} \t\t####{bcolors.ENDC}''')
    evaluator.evaluate(unlearner,predictor)

`�sO/,-748263990 | INFO | 3517115 - Created Configurable: erasure.evaluations.running.RunTime
�lT8/,-748263989 | INFO | 3517115 - Function: <function accuracy_score at 0x7f2f3bdde3a0>
`�sO/,-748263988 | INFO | 3517115 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
�nT8/,-748263976 | INFO | 3517115 - Function: <function accuracy_score at 0x7f2f3bdde3a0>
`�sO/,-748263975 | INFO | 3517115 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
@RC8/,-748263974 | INFO | 3517115 - Function: <function accuracy_score at 0x7f2f3bdde3a0>
`�sO/,-748263973 | INFO | 3517115 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
PTC8/,-748263972 | INFO | 3517115 - Function: <function accuracy_score at 0x7f2f3bdde3a0>
`�sO/,-748263971 | INFO | 3517115 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
`�sO/,-748263971 | INFO | 3517115 - Created Configurable: erasure.evaluations.measures.SaveValues
`�sO/,-748263970 

/NFSHOME/adangelo/LinkInference/erasure/unlearners/certified_graph_unlearners/IDEA.py:99: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(self.labels)


PARAMS CHANGE [tensor([ 4.1613e-03, -8.5233e-04,  3.0936e-04, -3.3698e-03, -4.4738e-04,
        -2.8269e-03,  4.1424e-03, -7.4017e-03,  7.2697e-04, -6.4575e-03,
        -1.0713e-03, -3.2154e-03, -1.4146e-03, -2.9489e-04, -9.2740e-03,
        -1.3836e-03, -2.0278e-02, -1.0444e-03,  3.6979e-03, -1.5262e-03,
        -2.7481e-03, -4.4610e-03, -1.7225e-03,  2.1515e-03,  2.0359e-03,
        -3.9537e-03, -6.9293e-03, -1.1044e-03,  3.3588e-03, -6.5928e-03,
        -3.3462e-03, -7.6298e-03, -3.8778e-03,  5.0248e-05, -3.8243e-03,
         4.1177e-04, -2.3112e-03, -5.2475e-03, -1.7266e-03, -4.6123e-03,
         4.4681e-03, -2.5450e-03,  3.5458e-03, -1.0449e-02,  8.9164e-03,
         3.3588e-03, -4.3590e-03, -3.6485e-03, -4.4788e-04, -8.2140e-03,
         3.5288e-03, -2.2919e-03,  6.6914e-03, -3.5294e-03, -7.2377e-03,
        -5.3851e-03,  8.6652e-03, -2.6033e-03,  1.1714e-02, -8.0577e-03,
        -6.1521e-03,  4.6007e-03,  6.3473e-04, -9.0733e-03,  4.5180e-03,
        -4.3373e-03, -3.6861e-03, -7